# Atividade 1 - Tratamento de Dados (Versão Simples)

Objetivo: ler o `detalhe_votacao.csv`, tratar os dados (texto sem acento, maiúsculo, datas no padrão do SGBD) e gerar o Excel final.

In [19]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

In [20]:
def remover_acentos(texto: str) -> str:
    normalizado = unicodedata.normalize('NFD', texto)
    return ''.join(c for c in normalizado if unicodedata.category(c) != 'Mn')


def normalizar_texto(valor):
    if pd.isna(valor):
        return None
    txt = str(valor).strip()
    txt = re.sub(r'[\n\r\t]+', ' ', txt)
    txt = re.sub(r'\s+', ' ', txt)
    txt = txt.lstrip('�')
    return remover_acentos(txt).upper()


def parse_data(valor):
    if pd.isna(valor):
        return None
    txt = str(valor).strip().lstrip('�')
    txt = re.sub(r'^[^\d]+', '', txt)
    dt = pd.to_datetime(txt, dayfirst=True, errors='coerce')
    return None if pd.isna(dt) else dt.strftime('%Y-%m-%d')

In [21]:
# 1) Ler arquivo
base = Path.cwd()
entrada = base / 'detalhe_votacao.csv'
saida = base / 'detalhe_votacao_tratado.xlsx'

# Arquivo está em ; e com possíveis problemas de acento
try:
    df = pd.read_csv(entrada, sep=';', dtype=str, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(entrada, sep=';', dtype=str, encoding='latin1')

df.columns = [c.strip() for c in df.columns]
for c in df.columns:
    df[c] = df[c].map(lambda x: x.strip() if isinstance(x, str) else x)

print('Linhas lidas:', len(df))
df.head(3)

Linhas lidas: 12459


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,...,QT_TOTAL_VOTOS_ANUL_SUBJUD,QT_VOTOS_NOMINAIS_ANUL_SUBJUD,QT_VOTOS_LEGENDA_ANUL_SUBJUD,QT_VOTOS_BRANCOS,QT_TOTAL_VOTOS_NULOS,QT_VOTOS_NULOS,QT_VOTOS_NULOS_TECNICOS,QT_VOTOS_ANULADOS_APU_SEP,HH_ULTIMA_TOTALIZACAO,DT_ULTIMA_TOTALIZACAO
0,30.07.2025,16:31:07,2024,2,Eleição Ordinária,1,619,Eleições Municipais 2024,06/10/2024,M,...,0,0,0,232,307,300,7,0,19:04:12,06/10/2024
1,30.07.2025,16:31:07,2024,2,Eleição Ordinária,1,619,Eleições Municipais 2024,06/10/2024,M,...,2,0,0,614,734,731,3,0,18:33:44,06/10/2024
2,30.07.2025,16:31:07,2024,2,Eleição Ordinária,1,619,Eleições Municipais 2024,06/10/2024,M,...,1,0,0,172,264,264,0,0,15:05:09,26/11/2024


In [22]:
# 2) Tratamento principal
colunas_data = [c for c in df.columns if c.startswith('DT_')]
colunas_numericas = [c for c in df.columns if c.startswith(('QT_', 'NR_', 'CD_', 'ANO_'))]
colunas_texto = [c for c in df.columns if c not in (colunas_data + colunas_numericas)]

for c in colunas_data:
    df[c] = df[c].map(parse_data)

for c in colunas_numericas:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')

for c in colunas_texto:
    df[c] = df[c].map(normalizar_texto)

# Enriquecimento
if {'QT_COMPARECIMENTO', 'QT_APTOS'}.issubset(df.columns):
    aptos = pd.to_numeric(df['QT_APTOS'], errors='coerce')
    comp = pd.to_numeric(df['QT_COMPARECIMENTO'], errors='coerce')
    df['PERC_COMPARECIMENTO'] = ((comp / aptos) * 100).round(2)

df = df.drop_duplicates()
df.head(3)

,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,...,QT_VOTOS_NOMINAIS_ANUL_SUBJUD,QT_VOTOS_LEGENDA_ANUL_SUBJUD,QT_VOTOS_BRANCOS,QT_TOTAL_VOTOS_NULOS,QT_VOTOS_NULOS,QT_VOTOS_NULOS_TECNICOS,QT_VOTOS_ANULADOS_APU_SEP,HH_ULTIMA_TOTALIZACAO,DT_ULTIMA_TOTALIZACAO,PERC_COMPARECIMENTO
0,2025-07-30,16:31:07,2024,2,ELEICAO ORDINARIA,1,619,ELEICOES MUNICIPAIS 2024,2024-10-06,M,...,0,0,232,307,300,7,0,19:04:12,2024-10-06,77.98
1,2025-07-30,16:31:07,2024,2,ELEICAO ORDINARIA,1,619,ELEICOES MUNICIPAIS 2024,2024-10-06,M,...,0,0,614,734,731,3,0,18:33:44,2024-10-06,79.87
2,2025-07-30,16:31:07,2024,2,ELEICAO ORDINARIA,1,619,ELEICOES MUNICIPAIS 2024,2024-10-06,M,...,0,0,172,264,264,0,0,15:05:09,2024-11-26,79.14


In [23]:
# 3) Exportar

df.to_excel(saida, index=False)
print('Arquivo gerado com sucesso:')
print(saida)
print('\nColunas finais:', len(df.columns))
print('Linhas finais:', len(df))

Arquivo gerado com sucesso:
/Users/caioacayabafurtado/Documents/GIT-HUB/arquivos-sptech/Materiais - Semestre5/Análise de dados/atividade1/detalhe_votacao_tratado.xlsx

Colunas finais: 48
Linhas finais: 12459
